In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

In [2]:
# Sample user-movie ratings
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4, 5],
    'movie': ['A', 'B', 'C', 'A', 'B', 'B', 'C', 'A', 'C'],
    'rating': [5, 4, 3, 5, 3, 4, 2, 3, 5]
}

df = pd.DataFrame(data)
df

,user_id,movie,rating
0,1,A,5
1,1,B,4
2,1,C,3
3,2,A,5
4,2,B,3
5,3,B,4
6,3,C,2
7,4,A,3
8,5,C,5


In [3]:
user_movie_matrix = df.pivot_table(index='user_id', columns='movie', values='rating').fillna(0)
print(user_movie_matrix)


movie      A    B    C
user_id               
1        5.0  4.0  3.0
2        5.0  3.0  0.0
3        0.0  4.0  2.0
4        3.0  0.0  0.0
5        0.0  0.0  5.0


In [4]:
# Convert to sparse matrix
sparse_matrix = csr_matrix(user_movie_matrix.values)

# Fit KNN on user data
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(sparse_matrix)


NearestNeighbors(algorithm='brute', metric='cosine')

In [5]:
# Let's pick user_id = 1 (index 0 in the matrix)
query_index = 0
distances, indices = model_knn.kneighbors(user_movie_matrix.iloc[query_index, :].values.reshape(1, -1), n_neighbors=3)

print("Similar users to User 1:")
for i in range(1, len(distances[0])):
    print(f"User {user_movie_matrix.index[indices[0][i]]} with distance {distances[0][i]:.2f}")


Similar users to User 1:
User 2 with distance 0.10
User 4 with distance 0.29


In [6]:
# Recommend movies that similar users liked and user 1 hasn't rated
target_user = user_movie_matrix.iloc[query_index]
similar_user_index = indices[0][1]  # first similar user
similar_user_ratings = user_movie_matrix.iloc[similar_user_index]

# Recommend items that the similar user liked (rating >= 4), and the target user hasn't rated
recommendations = similar_user_ratings[(similar_user_ratings >= 4) & (target_user == 0)].index.tolist()
print(f"Recommended movies for User 1: {recommendations}")


Recommended movies for User 1: []


In [11]:
import pandas as pd
import numpy as np

In [12]:
# Each row is a user, each column is a movie
data = {
    'Movie A': [5, 5, np.nan, 3, 1],
    'Movie B': [4, np.nan, 4, 3, 1],
    'Movie C': [3, 2, 5, np.nan, 1],
    'Movie D': [np.nan, 1, 5, 3, 2]
}
user_ratings = pd.DataFrame(data, index=['User1', 'User2', 'User3', 'User4', 'User5'])
print(user_ratings)


       Movie A  Movie B  Movie C  Movie D
User1      5.0      4.0      3.0      NaN
User2      5.0      NaN      2.0      1.0
User3      NaN      4.0      5.0      5.0
User4      3.0      3.0      NaN      3.0
User5      1.0      1.0      1.0      2.0


In [13]:
# Transpose to have users as columns for corr()
user_similarity = user_ratings.T.corr(method='pearson')
print(user_similarity)


       User1     User2  User3  User4     User5
User1    1.0  1.000000   -1.0    NaN       NaN
User2    1.0  1.000000    NaN    NaN -0.693375
User3   -1.0       NaN    1.0    NaN  0.500000
User4    NaN       NaN    NaN    NaN       NaN
User5    NaN -0.693375    0.5    NaN  1.000000


In [14]:
def recommend_movies(user_id, user_ratings, top_n=2):
    similarities = user_ratings.T.corrwith(user_ratings.loc[user_id], method='pearson')
    similarities = similarities.drop(user_id).dropna()
    
    # Weighted sum of ratings from similar users
    weighted_ratings = pd.Series(dtype=np.float64)
    for other_user, similarity in similarities.items():
        other_ratings = user_ratings.loc[other_user]
        weighted_ratings = weighted_ratings.add(other_ratings * similarity, fill_value=0)
    
    # Normalize by sum of similarities
    sum_similarities = similarities.abs().sum()
    predicted_ratings = weighted_ratings / sum_similarities

    # Filter out movies already rated
    unrated = user_ratings.loc[user_id].isna()
    recommendations = predicted_ratings[unrated].sort_values(ascending=False).head(top_n)
    
    return recommendations

recommendations = recommend_movies('User1', user_ratings)
print("Recommended movies for User1:")
print(recommendations)


Recommended movies for User1:
Movie D   -2.0
dtype: float64


C:\Users\z72w146\AppData\Roaming\Python\Python39\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\z72w146\AppData\Roaming\Python\Python39\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [20]:
import pandas as pd

# Load the dataset
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('ml-100k/u.data', sep='\t', names=column_names)

# Create a user-item ratings matrix
ratings_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating')
                                                                                                                                    

In [21]:
# Compute Pearson correlation between users
user_similarity = ratings_matrix.corr(method='pearson')


In [22]:
def recommend_movies(user_id, ratings_matrix, user_similarity, top_n=5):
    # Get the similarity scores for the target user
    sim_scores = user_similarity[user_id].drop(labels=[user_id])
    
    # Get the ratings of all users
    user_ratings = ratings_matrix.loc[user_id]
    
    # Identify movies not rated by the user
    unrated_movies = user_ratings[user_ratings.isnull()].index
    
    # Predict ratings for unrated movies
    predicted_ratings = {}
    for movie in unrated_movies:
        # Get ratings for the current movie
        movie_ratings = ratings_matrix[movie]
        
        # Compute the weighted sum of ratings
        weighted_sum = (movie_ratings * sim_scores).sum()
        sum_of_weights = sim_scores[movie_ratings.notnull()].sum()
        
        if sum_of_weights > 0:
            predicted_ratings[movie] = weighted_sum / sum_of_weights
    
    # Sort the predicted ratings
    recommended_movies = sorted(predicted_ratings.items(), key=lambda x: x[1], reverse=True)
    
    return recommended_movies[:top_n]


In [23]:
# Recommend top 5 movies for user with ID 1
recommendations = recommend_movies(1, ratings_matrix, user_similarity, top_n=5)
print("Recommended movies for User 1:")
for movie_id, rating in recommendations:
    print(f"Movie ID: {movie_id}, Predicted Rating: {rating:.2f}")


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [26]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Create your 5x5 matrix
np.random.seed(12)
m = np.random.randint(1, 101, size=(5, 5))

# Simulate explicit ratings (1–5, like Netflix)
m_ratings = np.ceil(m / 20).astype(int)

# Simulate unary data (1 = interaction, 0 = no interaction)
m_unary = (m > 50).astype(int)

# Introduce missing values (40% sparsity)
np.random.seed(42)
mask = np.random.choice([1, 0], size=(5, 5), p=[0.6, 0.4])  # 60% kept, 40% missing
m_ratings_sparse = m_ratings * mask  # Set missing entries to 0
m_unary_sparse = m_unary * mask

print("Sparse rating matrix (explicit, 1–5, 0 = missing):")
print(m_ratings_sparse)
print("\nSparse unary matrix (implicit, 1/0, 0 = missing/no interaction):")
print(m_unary_sparse)

# Function to compute co-rated cosine similarity
def co_rated_cosine_similarity(matrix):
    n_users = matrix.shape[0]
    sim_matrix = np.zeros((n_users, n_users))
    for u in range(n_users):
        for v in range(u, n_users):
            # Identify co-rated items (non-zero for both users)
            common_items = (matrix[u] != 0) & (matrix[v] != 0)
            if common_items.sum() > 0:  # At least 1 co-rated item
                ratings_u = matrix[u, common_items]
                ratings_v = matrix[v, common_items]
                norm_u = np.sqrt(np.sum(ratings_u ** 2))
                norm_v = np.sqrt(np.sum(ratings_v ** 2))
                if norm_u > 0 and norm_v > 0:
                    sim = np.dot(ratings_u, ratings_v) / (norm_u * norm_v)
                    sim_matrix[u, v] = sim
                    sim_matrix[v, u] = sim
                else:
                    sim_matrix[u, v] = 0
                    sim_matrix[v, u] = 0
            else:
                sim_matrix[u, v] = 0
                sim_matrix[v, u] = 0
    return sim_matrix

# Compute co-rated cosine similarity
sim_ratings = co_rated_cosine_similarity(m_ratings_sparse)
sim_unary = co_rated_cosine_similarity(m_unary_sparse)

# Convert to distance matrix
dist_ratings = 1 - sim_ratings
dist_unary = 1 - sim_unary
dist_ratings = np.clip(dist_ratings, 0, 2)
dist_unary = np.clip(dist_unary, 0, 2)

# Fit NearestNeighbors
model_knn_ratings = NearestNeighbors(metric='precomputed', algorithm='brute')
model_knn_ratings.fit(dist_ratings)

model_knn_unary = NearestNeighbors(metric='precomputed', algorithm='brute')
model_knn_unary.fit(dist_unary)

# Predict missing values
def predict_values(matrix, similarity, k=2):
    n_users, n_items = matrix.shape
    predictions = np.zeros_like(matrix, dtype=float)
    for u in range(n_users):
        # Find top k similar users (excluding self)
        sim_scores = similarity[u].copy()
        sim_scores[u] = -1  # Exclude self
        top_k_users = np.argsort(sim_scores)[-k:]  # Top k neighbors
        for i in range(n_items):
            # Predict rating/interaction for user u, item i
            sim_sum = sum(sim_scores[v] for v in top_k_users if matrix[v, i] != 0)
            if sim_sum > 0:
                pred = sum(sim_scores[v] * matrix[v, i] for v in top_k_users if matrix[v, i] != 0) / sim_sum
                predictions[u, i] = pred
    return predictions

# Predict ratings and interactions
pred_ratings = predict_values(m_ratings_sparse, sim_ratings, k=2)
pred_unary = predict_values(m_unary_sparse, sim_unary, k=2)

print("\nPredicted ratings (explicit, 0 = no prediction):")
print(np.round(pred_ratings, 2))
print("\nPredicted interactions (unary, 0 = no prediction):")
print(np.round(pred_unary, 2))

# Evaluate known values with MAE and RMSE
# Use original non-missing values from m_ratings and m_unary
actual_ratings = m_ratings[mask == 1].flatten()
predicted_ratings = pred_ratings[mask == 1].flatten()
actual_unary = m_unary[mask == 1].flatten()
predicted_unary = pred_unary[mask == 1].flatten()

mae_ratings = mean_absolute_error(actual_ratings[predicted_ratings != 0], predicted_ratings[predicted_ratings != 0])
rmse_ratings = np.sqrt(mean_squared_error(actual_ratings[predicted_ratings != 0], predicted_ratings[predicted_ratings != 0]))
mae_unary = mean_absolute_error(actual_unary[predicted_unary != 0], predicted_unary[predicted_unary != 0])
rmse_unary = np.sqrt(mean_squared_error(actual_unary[predicted_unary != 0], predicted_unary[predicted_unary != 0]))

print("\nEvaluation for explicit ratings (known values):")
print(f"MAE: {mae_ratings:.2f}")
print(f"RMSE: {rmse_ratings:.2f}")

print("\nEvaluation for unary data (known values):")
print(f"MAE: {mae_unary:.2f}")
print(f"RMSE: {rmse_unary:.2f}")

Sparse rating matrix (explicit, 1–5, 0 = missing):
[[4 0 0 1 1]
 [4 4 0 0 0]
 [3 0 0 5 2]
 [4 4 1 4 1]
 [0 5 2 4 2]]

Sparse unary matrix (implicit, 1/0, 0 = missing/no interaction):
[[1 0 0 0 0]
 [1 1 0 0 0]
 [1 0 0 1 0]
 [1 1 0 1 0]
 [0 1 0 1 0]]

Predicted ratings (explicit, 0 = no prediction):
[[4.   4.49 2.   4.   2.  ]
 [3.   5.   2.   4.5  2.  ]
 [4.   4.5  2.   4.   2.  ]
 [4.   4.49 2.   4.   2.  ]
 [3.5  4.   0.   5.   2.  ]]

Predicted interactions (unary, 0 = no prediction):
[[1. 1. 0. 1. 0.]
 [1. 1. 0. 1. 0.]
 [1. 1. 0. 1. 0.]
 [1. 1. 0. 1. 0.]
 [1. 1. 0. 1. 0.]]

Evaluation for explicit ratings (known values):
MAE: 0.78
RMSE: 1.07

Evaluation for unary data (known values):
MAE: 0.09
RMSE: 0.30
